In [13]:
!pip -q install --upgrade ultralytics==8.3.177 opencv-python PyYAML kaggle


In [14]:
from ultralytics.utils import SETTINGS
SETTINGS.update({'datasets_dir': '/content'})
print("Ultralytics datasets_dir:", SETTINGS['datasets_dir'])


Ultralytics datasets_dir: /content


In [15]:
from google.colab import files
import os, shutil

uploaded = files.upload()  # kaggle.json'ı seç
assert 'kaggle.json' in uploaded, "kaggle.json yüklenmedi."
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Kaggle kimlik dosyası kuruldu.")


Saving kaggle.json to kaggle.json
Kaggle kimlik dosyası kuruldu.


In [17]:
!kaggle datasets download -d smaildurcan/turkish-license-plate-dataset -p /content/data/raw --unzip


Dataset URL: https://www.kaggle.com/datasets/smaildurcan/turkish-license-plate-dataset
License(s): CC0-1.0
 99% 2.88G/2.90G [00:06<00:00, 278MB/s]
100% 2.90G/2.90G [00:06<00:00, 456MB/s]


In [18]:
from pathlib import Path
import shutil, random, yaml, re

RAW_BASE = Path("/content/data/raw")
PROC_IMG = Path("/content/data/processed/images")
PROC_LBL = Path("/content/data/processed/labels")

# Temizle
for p in [PROC_IMG, PROC_LBL]:
    shutil.rmtree(p, ignore_errors=True)

# 1) Tüm img ve label dosyalarını nerede olursa olsun tara
IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp",".JPG",".PNG"}
all_imgs = [p for p in RAW_BASE.rglob("*") if p.suffix in IMG_EXTS]
all_lbls = [p for p in RAW_BASE.rglob("*.txt")]

print(f"[INFO] raw içinde bulunan: {len(all_imgs)} görüntü, {len(all_lbls)} label")

# 2) Label doğrulayıcı (YOLO formatına kabaca uygun mu?)
yo_re = re.compile(r"^\s*\d+(\s+([0-1]?(\.\d+)?)){4}\s*$")
def is_yolo_label_file(txt: Path) -> bool:
    try:
        lines = [ln for ln in txt.read_text().splitlines() if ln.strip()]
        if not lines:
            return False
        # En azından ilk satır 5 alanlı ve [0..1] aralığında olsun
        return bool(yo_re.match(lines[0]))
    except Exception:
        return False

# 3) Aynı isimli (stem) birden fazla txt olabilir; 'labels' geçen yolu tercih et
from collections import defaultdict
label_by_stem = defaultdict(list)
for t in all_lbls:
    if is_yolo_label_file(t):
        label_by_stem[t.stem].append(t)

def pick_label(stem):
    cand = label_by_stem.get(stem, [])
    if not cand:
        return None
    # Önce yolu 'labels' içereni, yoksa en kısa yolu seç
    cand_sorted = sorted(cand, key=lambda p: ("/labels/" not in str(p).replace("\\","/"), len(str(p))))
    return cand_sorted[0]

# 4) Eşleşen çiftleri oluştur
pairs = []
for img in all_imgs:
    lab = pick_label(img.stem)
    if lab is not None:
        pairs.append((img, lab))

print(f"[INFO] eşleşen image/label çifti: {len(pairs)}")
if not pairs:
    # Teşhis için kısa bir listing göster
    print("[DIAG] Örnek klasörler:")
    for p in list(RAW_BASE.glob("*"))[:10]:
        print(" -", p)
    raise SystemExit("Eşleşen çift bulunamadı. Zip yapısını kontrol et (farklı uzantı ya da isimlendirme olabilir).")

# 5) Split ve kopyala
random.seed(42); random.shuffle(pairs)
n = len(pairs); n_tr = int(n*0.7); n_val = int(n*0.2)
train, val, test = pairs[:n_tr], pairs[n_tr:n_tr+n_val], pairs[n_tr+n_val:]

def cp(pairs, split):
    (PROC_IMG/split).mkdir(parents=True, exist_ok=True)
    (PROC_LBL/split).mkdir(parents=True, exist_ok=True)
    for img, lab in pairs:
        shutil.copy2(img, PROC_IMG/split/img.name)
        shutil.copy2(lab, PROC_LBL/split/lab.name)

cp(train,"train"); cp(val,"val"); cp(test,"test")

# 6) dataset.yaml (ABS path)
ds = {
    "train": str(PROC_IMG/"train"),
    "val":   str(PROC_IMG/"val"),
    "test":  str(PROC_IMG/"test"),
    "nc": 1,
    "names": ["license_plate"]
}
Path("/content/data/dataset.yaml").write_text(yaml.safe_dump(ds, sort_keys=False))

# 7) Sayımları yaz
def cnt_imgs(p): return sum(1 for f in Path(p).glob("*.*"))
print("\n--- dataset.yaml ---\n", Path("/content/data/dataset.yaml").read_text())
print("Counts:",
      "\n train images:", cnt_imgs(PROC_IMG/'train'), "labels:", len(list((PROC_LBL/'train').glob('*.txt'))),
      "\n val   images:", cnt_imgs(PROC_IMG/'val'),   "labels:", len(list((PROC_LBL/'val').glob('*.txt'))),
      "\n test  images:", cnt_imgs(PROC_IMG/'test'),  "labels:", len(list((PROC_LBL/'test').glob('*.txt'))))


[INFO] raw içinde bulunan: 1955 görüntü, 3910 label
[INFO] eşleşen image/label çifti: 1955

--- dataset.yaml ---
 train: /content/data/processed/images/train
val: /content/data/processed/images/val
test: /content/data/processed/images/test
nc: 1
names:
- license_plate

Counts: 
 train images: 1368 labels: 1368 
 val   images: 391 labels: 391 
 test  images: 196 labels: 196


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [19]:
!yolo detect train \
  data=/content/data/dataset.yaml \
  model=yolov8n.pt \
  epochs=60 imgsz=640 batch=-1 device=0 workers=8 \
  patience=10


New https://pypi.org/project/ultralytics/8.3.178 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.177 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optim

In [20]:
# VAL split
!yolo detect val \
  model=runs/detect/train/weights/best.pt \
  data=/content/data/dataset.yaml \
  device=0


Ultralytics 8.3.177 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1554.1±362.9 MB/s, size: 4626.6 KB)
val: Scanning /content/data/processed/labels/val.cache... 391 images, 0 backgrounds, 0 corrupt: 100% 391/391 [00:00<?, ?it/s]
val: /content/data/processed/images/val/1.jpg: corrupt JPEG restored and saved
val: /content/data/processed/images/val/208.jpg: corrupt JPEG restored and saved
val: /content/data/processed/images/val/211.jpg: corrupt JPEG restored and saved
val: /content/data/processed/images/val/219.jpg: corrupt JPEG restored and saved
val: /content/data/processed/images/val/225.jpg: corrupt JPEG restored and saved
val: /content/data/processed/images/val/226.jpg: corrupt JPEG restored and saved
val: /content/data/processed/images/val/239.jpg: corrupt JPEG restored and saved
val: /content/data/processed/images/val/254.jpg

In [21]:
# TEST split (dataset.yaml'da test varsa çalışır)
from pathlib import Path
import yaml, subprocess, sys

cfg = yaml.safe_load(Path("/content/data/dataset.yaml").read_text())
test_path = cfg.get("test")
if test_path and Path(test_path).exists():
    print("Test split bulundu, doğrulama başlatılıyor...")
    cmd = "yolo detect val model=runs/detect/train/weights/best.pt data=/content/data/dataset.yaml device=0 split=test"
    print(cmd)
    subprocess.run(cmd.split(), check=True)
else:
    print("Test split tanımlı değil; atlandı.")


Test split bulundu, doğrulama başlatılıyor...
yolo detect val model=runs/detect/train/weights/best.pt data=/content/data/dataset.yaml device=0 split=test


In [23]:
from google.colab import drive, files
import glob, os, hashlib, shutil

cands = sorted(glob.glob("runs/detect/train*/weights/best.pt"), key=os.path.getmtime)
assert cands, "best.pt bulunamadı."
best = cands[-1]
print("Bulunan model:", best, f"({os.path.getsize(best)/1024/1024:.2f} MB)")
print("SHA256:", hashlib.sha256(open(best,'rb').read()).hexdigest())

drive.mount('/content/drive')
out_dir = "/content/drive/MyDrive/tr-yolo-plate-recognition/models/final"
os.makedirs(out_dir, exist_ok=True)
dst = os.path.join(out_dir, "best_tr_plate_model.pt")
shutil.copy2(best, dst)
print("Drive'a kopyalandı:", dst)

# Tarayıcı ile indirmek istersen aşağıyı aç:
# files.download(best)


Bulunan model: runs/detect/train/weights/best.pt (5.94 MB)
SHA256: bd99e0e7dbb0cb63cdaead9f41da266ce5725d08079635367a16e54d93b53bc4
Mounted at /content/drive
Drive'a kopyalandı: /content/drive/MyDrive/tr-yolo-plate-recognition/models/final/best_tr_plate_model.pt
